# Day 032 Project Solution — Secure Config Module

A `SecureConfig` class loaded from a .env text string with validation, safe logging, and fail-fast access.

In [ ]:
def parse_dotenv(text: str) -> dict:
    result = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, _, value = line.partition("=")
        key   = key.strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in ('"', "'"):
            value = value[1:-1]
        result[key] = value
    return result


def mask_secret(value, show_chars: int = 4) -> str:
    s = str(value)
    if len(s) <= show_chars:
        return "***"
    return s[:show_chars] + "***" 


def validate_config(config: dict, required_keys: list) -> list:
    return [k for k in required_keys if config.get(k) is None]


def safe_log_config(config: dict, secret_keys: list) -> dict:
    return {
        k: mask_secret(str(v)) if k in secret_keys else v
        for k, v in config.items()
    }


class SecureConfig:
    def __init__(self, defaults: dict | None = None):
        self._config: dict = dict(defaults or {})

    def load_dict(self, mapping: dict) -> "SecureConfig":
        self._config.update(mapping)
        return self

    def get(self, key: str, default=None):
        return self._config.get(key, default)

    def require(self, key: str) -> str:
        val = self._config.get(key)
        if val is None:
            raise KeyError(f"Required config key not found: '{key}'")
        return str(val)

    def validate(self, required_keys: list) -> list:
        return validate_config(self._config, required_keys)

    def masked_dict(self, secret_keys: list) -> dict:
        return safe_log_config(self._config, secret_keys)

## Action 1 — Parse .env and Build SecureConfig

In [ ]:
ENV_TEXT = """
# API credentials
API_KEY=sk-demo-key-12345678
SLACK_WEBHOOK=https://hooks.slack.com/services/T123/B456/secrettoken

# Database
DB_HOST=localhost
DB_PORT=5432
DB_NAME=myapp

# Model
MODEL_NAME=llama3.2
DEBUG=false
"""

SECRET_KEYS   = ['API_KEY', 'SLACK_WEBHOOK']
REQUIRED_KEYS = ['API_KEY', 'DB_HOST', 'DB_NAME', 'MODEL_NAME']

cfg = SecureConfig(defaults={'MODEL_NAME': 'llama3.2', 'DEBUG': 'false'})
cfg.load_dict(parse_dotenv(ENV_TEXT))

print(f'Loaded {len(cfg._config)} config keys')
print('Keys:', list(cfg._config.keys()))

## Action 2 — Validate and Log Safely

In [ ]:
missing = cfg.validate(REQUIRED_KEYS)
if missing:
    raise EnvironmentError(f'Missing required config: {missing}')
print('\u2705 All required keys present')

safe = cfg.masked_dict(SECRET_KEYS)
print('\nConfig (safe for logging):')
for k, v in safe.items():
    print(f'  {k:20} = {v}')

## Action 3 — Use Config and Demonstrate Fail-Fast

In [ ]:
db_host = cfg.require('DB_HOST')
db_name = cfg.require('DB_NAME')
model   = cfg.require('MODEL_NAME')
api_key = cfg.require('API_KEY')

print(f'DB:    {db_host}/{db_name}')
print(f'Model: {model}')
print(f'Key:   {mask_secret(api_key)}')

# Demonstrate fail-fast on a missing required key
try:
    cfg.require('NONEXISTENT_KEY')
except KeyError as e:
    print(f'\nFail-fast: {e}')

# Verify functions work on their own too
assert parse_dotenv('KEY=val\n')   == {'KEY': 'val'}
assert mask_secret('sk-abc123')      == 'sk-a***'
assert validate_config({}, ['K'])    == ['K']
assert safe_log_config({'K': 'v'}, ['K']) == {'K': '***'}

print('\nConfig complete!')